# Does the study survive a realistic baseline and realistic noise?

**What this tests.** The landed notebook (`final/roi-vs-mroi-metric-selection.ipynb`)
runs against a DGP that flatters the model in two ways:

1. **The baseline is drawn from the model's own basis.** `n_knots_mu_t=8` generates
   `mu_t` from an 8-knot spline and `build_model_spec` fits an 8-knot spline, so the
   true time-varying baseline is exactly representable by the fitted one.
2. **The residual is iid**, giving a measured DGP R^2 of 0.9958 with essentially zero
   autocorrelation (-0.008) and zero cross-geo correlation (-0.010).

Neither holds in real data. This notebook rebuilds the DGP without those two
advantages and re-runs the study's three headline findings against it.

**This notebook does not modify the landed one, or the modules it depends on.**
Everything here lives in `realistic_baseline.py`, which *subclasses*
`GeoMediaDataSimulator` rather than editing it. Section 1 asserts that.

**What changes, precisely** (see the `realistic_baseline` module docstring):

- `mu_t` becomes `trend + phase-shifted seasonality + common AR(1) shock`, drawn
  from no spline basis. The seasonal term is a `sin` where the observed seasonality
  control is a `cos`, plus a second harmonic -- unspanned by the control, but
  orthogonal to it over whole years, so it adds realism without introducing
  omitted-variable bias (which would break *both* prior variants for reasons
  unrelated to saturation, confounding the study).
- `eps_gt` becomes a per-geo AR(1) with Student-t innovations, replacing the iid draw.

The two noise dials are deliberately non-overlapping: all cross-geo-correlated
structure lives in `mu_t`, all geo-idiosyncratic structure in `eps_gt`.

**Why cross-geo correlation is in scope at all.** Unmodeled drivers -- competitor
activity, category demand, national promos, stockouts -- are mostly national or
regional, not geo-idiosyncratic. That distinction matters more than the noise
*magnitude*: iid geo noise averages down at `1/sqrt(20)` (a 4.5x reduction), so a geo
hierarchy disposes of it nearly for free, while a common shock gets no such reduction
and competes with media for explanatory power the way a national flight does. Leaving
the noise iid tests the model in the one regime where its main strength is unopposed.

**Alpha is held at the study's usual 0.8 throughout.** Testing alpha=0.2 is a separate
question, deferred until this DGP is trusted.

## 0. Setup

In [ ]:
import os
import sys
import time
import warnings

STUDY_DIR = next(
    d for d in (os.path.abspath(os.getcwd()),
                os.path.abspath(os.path.join(os.getcwd(), '..')))
    if os.path.exists(os.path.join(d, 'data_simulator.py')))
if STUDY_DIR not in sys.path:
  sys.path.insert(0, STUDY_DIR)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from meridian.model import model

from data_simulator import SimulationConfig
from model_utils import build_comparison_table
from model_utils import build_model_spec
import realistic_baseline as rb

warnings.filterwarnings('ignore')
OUT_DIR = os.path.join(STUDY_DIR, 'fitted_models', 'realistic_baseline')
os.makedirs(OUT_DIR, exist_ok=True)

VARIANTS = ['default', 'ec_alpha_only']
MCMC_KWARGS = dict(n_chains=2, n_adapt=500, n_burnin=500, n_keep=1000, seed=1)
SIM_SEED = 0
N_KNOTS = 8
PRIMARY, COMPARISON = 'ec9', 'ec11'

# `None` = the baseline rewrite with no extra noise, isolating the effect of the
# baseline change alone. The rest sweep the oracle R^2 down to the ~0.80 that a
# real geo MMM sees.
R2_TARGETS = [None, 0.95, 0.90, 0.80]

# Identical to the landed notebook except `n_knots_mu_t` is GONE -- that field is
# precisely the "DGP drawn from the model's basis" problem being fixed here.
BASE_OVERRIDES = {
    'target_audience_pop_frac': {'TV': 0.60, 'Display': 0.50, 'Social': 0.15},
    'current_reach_frac': {'TV': 0.10, 'Display': 0.5, 'Social': 0.9},
    'frequency_range': {'TV': (1, 2), 'Display': (1, 5), 'Social': (7.0, 8.0)},
    'target_roi': {'TV': 8.0, 'Display': 5.0, 'Social': 3.0},
    'max_lag': 13,
    'n_times': 156,
    'roi_ec_elasticity': -0.3,
    'roi_alpha_elasticity': 0.3,
    'adstock_retention_range': {
        'TV': (0.8, 0.8), 'Display': (0.0, 0.3), 'Social': (0.1, 0.4),
    },
}
TARGET_EC_M = {'ec9': 9.0, 'ec11': 11.069}

# Read the demo dataset from a local cache when present. The landed notebook
# fetches it over HTTPS on every run; that makes execution depend on the
# kernel's certificate store, which is a needless failure mode for a notebook
# whose point is a 30-minute unattended fit sweep.
REAL_CSV_URL = (
    'https://raw.githubusercontent.com/google/meridian/refs/heads/main/'
    'meridian/data/simulated_data/csv/geo_media_rf.csv'
)
REAL_CSV_CACHE = os.path.join(STUDY_DIR, '.cache_geo_media_rf.csv')
if not os.path.exists(REAL_CSV_CACHE):
  pd.read_csv(REAL_CSV_URL).to_csv(REAL_CSV_CACHE, index=False)
real_df = pd.read_csv(REAL_CSV_CACHE)

print('study dir:', STUDY_DIR)
print('real data:', real_df.shape)

## 1. Isolation check

The landed notebook must be bit-for-bit unaffected by this one. `realistic_baseline`
subclasses rather than edits, so this should hold by construction -- assert it rather
than assume it.

In [ ]:
landed = rb.assert_landed_path_unchanged(real_df, {**BASE_OVERRIDES, 'n_knots_mu_t': 8,
                                                   'saturation_frequency': {
                                                       'TV': 3.2522, 'Display': 4.0,
                                                       'Social': 4.0}})
print('landed path reproduces bit-identically')
print('  true ec_m ', np.round(landed['true_ec_m'], 3))
print('  true roi_m', np.round(landed['true_roi_m'], 3))
print('  kpi checksum', f"{landed['kpi_checksum']:.6e}")

## 2. Build the realistic-baseline scenarios

Same `saturation_frequency` solve as the landed notebook: `ec_m` is linear in it, so
one baseline run calibrates both scenarios. Media execution must stay bit-identical
between `ec9` and `ec11` for Section 6's invariance test to be a clean single-variable
comparison.

**One thing that does not carry over:** true `roi_m` differs from the landed
notebook's, because `simulate_intercepts`/`simulate_coefficients` now consume a
different amount of randomness, so Display's and Social's `alpha_m` draws differ, and
`roi_alpha_elasticity=0.3` ties every channel's true ROI to the alpha geometric mean.
TV's alpha is still pinned at exactly 0.8 and TV's true `ec_m` is still exactly 9.0.
Compare fitted-vs-true *within* a run; never compare ROI levels across notebooks.

In [ ]:
def build(saturation_frequency_tv=None, target_r2=None, realism=None):
  overrides = dict(BASE_OVERRIDES)
  if saturation_frequency_tv is not None:
    overrides['saturation_frequency'] = {
        'TV': saturation_frequency_tv, 'Display': 4.0, 'Social': 4.0}
  np.random.seed(SIM_SEED)
  tf.random.set_seed(SIM_SEED)
  cfg = SimulationConfig.from_dict(overrides)
  sim, data, gt = rb.build_real_augmented_realistic(
      cfg, real_df,
      rf_source_map={'TV': 'Channel3'},
      plain_source_map={'Display': 'Channel2', 'Social': 'Channel1'},
      realism=realism, target_oracle_r2=target_r2, n_knots=N_KNOTS,
  )
  return cfg, sim, data, gt


_, sim_base, _, _ = build()
EC_BASE, SF_BASE = float(sim_base.ec_m.numpy()[0]), 4.0
SATURATION_FREQ = {tag: SF_BASE * target / EC_BASE
                   for tag, target in TARGET_EC_M.items()}

scen = {}
for tag in TARGET_EC_M:
  cfg, sim, data, gt = build(SATURATION_FREQ[tag])
  scen[tag] = dict(config=cfg, sim=sim, data=data, ground_truth=gt,
                   true_roi=np.asarray(gt['roi_m']), true_ec=sim.ec_m.numpy(),
                   true_alpha=sim.alpha_m.numpy())
  assert abs(float(sim.ec_m.numpy()[0]) - TARGET_EC_M[tag]) < 0.05
  assert abs(float(sim.alpha_m.numpy()[0]) - 0.8) < 1e-6, 'TV alpha must stay 0.8'

channels = list(scen[PRIMARY]['config'].channel_names)
delta = np.abs(scen['ec9']['sim'].impression_gtm.numpy()
               - scen['ec11']['sim'].impression_gtm.numpy()).max()
assert delta == 0, f'media execution differs between scenarios ({delta})'
print(f'media execution identical across scenarios: max |diff| = {delta}')
print('true alpha_m:', np.round(scen[PRIMARY]['true_alpha'], 3))
print('true ec_m   :', np.round(scen[PRIMARY]['true_ec'], 3))
print('true roi_m  :', np.round(scen[PRIMARY]['true_roi'], 3))

## 3. What the baseline rewrite actually did

Two things to establish before any fit:

1. The fitted 8-knot spline can no longer represent the true `mu_t`. Under the landed
   DGP it could represent it *exactly*, by construction.
2. The unexplainable part of the KPI is now autocorrelated in time and correlated
   across geos, rather than white noise.

In [ ]:
sim = scen[PRIMARY]['sim']
mu = sim.mu_t.numpy()
basis = rb._spline_basis(len(mu), N_KNOTS)
mu_spline_fit = mu - rb._spline_residual(mu, N_KNOTS)

fig, axes = plt.subplots(2, 1, figsize=(13, 6.4), sharex=True)
axes[0].plot(mu, color='k', lw=1.8, label='true mu_t (realistic DGP)')
axes[0].plot(mu_spline_fit, color='#c1440e', lw=2.0, ls='--',
             label=f'best {N_KNOTS}-knot spline fit (what the model can represent)')
axes[0].set_ylabel('mu_t'); axes[0].legend(loc='upper right')
axes[0].set_title('The fitted baseline can no longer reach the true one')
axes[1].plot(mu - mu_spline_fit, color='#0b7285', lw=1.4)
axes[1].axhline(0, color='k', lw=1)
axes[1].set_ylabel('residual'); axes[1].set_xlabel('week')
axes[1].set_title('What the spline misses -- common across every geo, so the '
                  'hierarchy cannot average it away')
plt.tight_layout()

diag_rows = []
for target in R2_TARGETS:
  _, s, _, _ = build(SATURATION_FREQ[PRIMARY], target_r2=target)
  d = rb.baseline_diagnostics(s, N_KNOTS)
  d['target'] = 'none' if target is None else f'{target:.2f}'
  diag_rows.append(d)
diagnostics = pd.DataFrame(diag_rows).set_index('target')[[
    'oracle_r2', 'dgp_r2_eps_only', 'noise_scale', 'media_share_pct',
    'spline_explains_mu_pct', 'spline_explains_mu_shock_pct',
    'resid_lag1_autocorr', 'unexplained_cross_geo_corr', 'clipped_frac_pct',
]].round(3)
diagnostics.to_csv(os.path.join(OUT_DIR, 'baseline_diagnostics.csv'))
diagnostics

**How to read the diagnostics.**

- `oracle_r2` is the R^2 ceiling for Meridian's *mean structure* handed the true media
  contribution: geo intercepts, geo-varying control coefficients, an 8-knot common time
  spline. No fitted model can beat it. This is the number to quote, and the one the
  calibration targets -- unlike `1 - var(eps)/var(kpi)`, it charges the DGP for baseline
  misspecification rather than only for residual variance. The landed DGP's 0.9958 is
  the `dgp_r2_eps_only` column's analogue, shown alongside for comparability.
- The `target=none` row is the **baseline rewrite alone, no extra noise**. Its oracle
  R^2 shows how much of the landed notebook's apparent fit quality was the DGP handing
  the model its own basis.
- `spline_explains_mu_shock_pct` is why `mu_ar1_phi` is 0.6 rather than something more
  obviously "persistent": an AR(1)'s power sits at low frequencies, which is exactly
  what a spline absorbs, so a high phi produces a shock the model can largely fit
  anyway.
- `clipped_frac_pct` is the share of geo-weeks where the baseline hits the
  `max(..., 0)` floor in `generate_kpi_and_revenue`. It rises with noise. Real KPIs are
  also non-negative, so this is not purely an artifact, but it is a nonlinearity the
  fitted model does not have -- watch it at the lowest R^2.

## 4. Recovery across the R^2 sweep

`default` vs `ec_alpha_only`, primary scenario (`ec9`, true TV `ec_m` = 9.0), at each
noise level. Eight fits, roughly 20 minutes.

Reporting **point errors alongside HDI coverage marks**, per the landed notebook's own
warning: coverage checkmarks reward imprecision, and as noise rises the intervals widen,
so a ✓ becomes progressively less impressive.

In [ ]:
def fit(data, sim, config, variant):
  spec = build_model_spec(variant, sim, config, media_prior_type='roi',
                          knots=N_KNOTS)
  mmm = model.Meridian(input_data=data, model_spec=spec)
  mmm.sample_prior(500)
  mmm.sample_posterior(**MCMC_KWARGS)
  return mmm


def tv_posterior(mmm, param, idx=0):
  return float(mmm.inference_data.posterior[param].values[..., idx].mean())


fitted, sweep_rows = {}, []
for target in R2_TARGETS:
  label = 'none' if target is None else f'{target:.2f}'
  cfg, sim, data, gt = build(SATURATION_FREQ[PRIMARY], target_r2=target)
  achieved = rb.oracle_r2(sim, N_KNOTS)
  true_ec, true_roi = sim.ec_m.numpy(), np.asarray(gt['roi_m'])
  for variant in VARIANTS:
    t0 = time.time()
    mmm = fit(data, sim, cfg, variant)
    fitted[(label, variant)] = mmm
    ec, roi = tv_posterior(mmm, 'ec_m'), tv_posterior(mmm, 'roi_m')
    sweep_rows.append({
        'target_r2': label, 'oracle_r2': round(achieved, 3), 'variant': variant,
        'true_ec_m': round(float(true_ec[0]), 3), 'fitted_ec_m': round(ec, 3),
        'ec_err_pct': round((ec / true_ec[0] - 1) * 100, 1),
        'true_roi_m': round(float(true_roi[0]), 3), 'fitted_roi_m': round(roi, 3),
        'roi_err_pct': round((roi / true_roi[0] - 1) * 100, 1),
    })
    print(f'{label}/{variant}: {time.time() - t0:.0f}s  '
          f'ec_m {ec:.2f}  roi_m {roi:.2f}', flush=True)

sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(os.path.join(OUT_DIR, 'r2_sweep.csv'), index=False)
sweep

In [ ]:
# Full recovery table at the most realistic noise level, with HDI marks.
cfg, sim, data, gt = build(SATURATION_FREQ[PRIMARY], target_r2=R2_TARGETS[-1])
comparison = build_comparison_table(
    {v: fitted[(f'{R2_TARGETS[-1]:.2f}', v)] for v in VARIANTS},
    cfg, sim, gt, hdi_prob=0.9)
comparison.to_csv(os.path.join(OUT_DIR, 'comparison_at_lowest_r2.csv'), index=False)
print(f'parameter recovery at oracle R^2 = {R2_TARGETS[-1]}')
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, col, title in [(axes[0], 'ec_err_pct', 'TV ec_m error vs. noise level'),
                       (axes[1], 'roi_err_pct', 'TV roi_m error vs. noise level')]:
  for variant, color in [('default', '#c1440e'), ('ec_alpha_only', '#0b7285')]:
    sub = sweep.query('variant == @variant')
    ax.plot(sub['oracle_r2'], sub[col], 'o-', color=color, lw=2.0, label=variant)
  ax.axhline(0, color='k', lw=1)
  ax.set_xlabel('oracle R^2 (higher = cleaner data)')
  ax.set_ylabel('error vs. truth (%)')
  ax.set_title(title); ax.legend(); ax.invert_xaxis()
plt.tight_layout()

## 5. Does the invariance result survive?

The study's strongest claim: two scenarios differing in exactly one thing -- TV's true
`ec_m` -- with bit-identical media execution. When the truth moves, does the posterior
follow? Under the landed DGP the `default` fit absorbed 7% of the move and the informed
prior 99%.

This is the finding most likely to survive added noise, because it is a *ratio of two
fits under identical noise* rather than a level. It is also interval-free by
construction, which matters given the open TODO that the mROI sweep still lacks credible
intervals.

Run at both ends of the sweep: the baseline rewrite alone, and the most realistic noise
level. Four more fits, roughly 10 minutes.

In [ ]:
invariance_rows = []
for target in [None, R2_TARGETS[-1]]:
  label = 'none' if target is None else f'{target:.2f}'
  ec_by_variant = {}
  for tag in (COMPARISON, PRIMARY):
    cfg, sim, data, gt = build(SATURATION_FREQ[tag], target_r2=target)
    for variant in VARIANTS:
      key = (label, variant, tag)
      if tag == PRIMARY and (label, variant) in fitted:
        mmm = fitted[(label, variant)]
      else:
        t0 = time.time()
        mmm = fit(data, sim, cfg, variant)
        print(f'{key}: {time.time() - t0:.0f}s', flush=True)
      ec_by_variant.setdefault(variant, {})[tag] = tv_posterior(mmm, 'ec_m')

  truth_move = (TARGET_EC_M[PRIMARY] / TARGET_EC_M[COMPARISON] - 1) * 100
  for variant, vals in ec_by_variant.items():
    move = (vals[PRIMARY] / vals[COMPARISON] - 1) * 100
    invariance_rows.append({
        'noise_level': label, 'source': variant,
        f'ec_m@{COMPARISON}': round(vals[COMPARISON], 3),
        f'ec_m@{PRIMARY}': round(vals[PRIMARY], 3),
        'pct_change': round(move, 1),
        'tracks_truth': round(move / truth_move, 2),
    })

invariance = pd.DataFrame(invariance_rows)
invariance.to_csv(os.path.join(OUT_DIR, 'invariance_realistic.csv'), index=False)
print(f'TV true ec_m moves {truth_move:+.1f}% between scenarios '
      '(tracks_truth 1.0 = followed one-for-one, 0.0 = ignored entirely)')
invariance

## 6. Verdict

Read three things off the tables above, in this order:

1. **Section 3, `target=none` row.** How much of the landed notebook's fit quality was
   the DGP handing the model its own spline basis? That is the cost of the
   `n_knots_mu_t` shortcut alone, before any noise is added.
2. **Section 4.** Does the `ec_m` failure persist as the oracle R^2 falls to ~0.80, and
   does the informed prior keep its advantage? Watch for the informed prior degrading
   too -- the outline already records it losing 19-29% mROI accuracy in a weak-signal
   variant, and a weak-signal result is a signal-to-noise caveat, not a saturation
   finding. If **both** variants collapse at the lowest R^2, TV is simply not
   identified there and that level should not be quoted as evidence about priors.
3. **Section 5.** If `tracks_truth` still separates `default` from `ec_alpha_only` at
   the realistic noise level, the study's strongest claim is noise-robust and belongs on
   the ARF slide as the lead, exactly as the outline's contingency plan suggests.

**Sequencing note for the talk (Tue Aug 4).** The outline's Section 5 numbers all come
from the landed notebook, which uses the flattering DGP. If this notebook materially
moves them, that is a talk-content decision, not just a technical one. The
recommendation stands: keep the landed notebook as the headline and present this as the
robustness section, unless Section 4 below shows the headline numbers are noise-fragile.